# Content Monetization Modeler

This notebook explains the complete machine learning workflow used to predict YouTube ad revenue. The style is intentionally simple and interview-friendly.

## 1. Problem Statement

The aim is to predict `ad_revenue_usd` using video performance and contextual features such as views, likes, comments, watch time, subscribers, category, device, and country. Since revenue is a continuous value, this is a regression problem.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
DATA_PATH = Path('../data/raw/youtube_ad_revenue_dataset.csv')
df = pd.read_csv(DATA_PATH)
df.head()

## 2. Dataset Understanding

In [ ]:
print('Shape:', df.shape)
display(df.dtypes.to_frame('dtype'))
display(df.describe().T)

The dataset has around 122k rows. Each row represents one video record with performance metrics and a target revenue value.

## 3. Missing Values and Duplicates

In [ ]:
missing = df.isna().sum().to_frame('missing_count')
missing['missing_percent'] = (missing['missing_count'] / len(df) * 100).round(2)
display(missing)
print('Duplicate rows:', df.duplicated().sum())

The main missing values are in numeric engagement columns. Median imputation is used because revenue and engagement features can be skewed by high-performing videos.

## 4. Target Variable Analysis

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df['ad_revenue_usd'], bins=50, color='#2f7d6d')
plt.title('Ad Revenue Distribution')
plt.xlabel('Ad Revenue USD')
plt.ylabel('Video Count')
plt.show()
df['ad_revenue_usd'].describe()

## 5. Univariate and Bivariate Analysis

In [ ]:
numeric_cols = ['views', 'likes', 'comments', 'watch_time_minutes', 'video_length_minutes', 'subscribers', 'ad_revenue_usd']
df[numeric_cols].hist(figsize=(12, 8), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(df['watch_time_minutes'], df['ad_revenue_usd'], alpha=0.2, s=5)
axes[0].set_title('Watch Time vs Revenue')
axes[0].set_xlabel('Watch Time Minutes')
axes[0].set_ylabel('Ad Revenue USD')
axes[1].scatter(df['likes'], df['ad_revenue_usd'], alpha=0.2, s=5)
axes[1].set_title('Likes vs Revenue')
axes[1].set_xlabel('Likes')
plt.tight_layout()
plt.show()

## 6. Category, Device, Country, and Subscriber Analysis

In [ ]:
for col in ['category', 'device', 'country']:
    display(df.groupby(col)['ad_revenue_usd'].mean().sort_values(ascending=False).to_frame('avg_revenue'))

subscriber_bins = pd.qcut(df['subscribers'], q=4, duplicates='drop')
display(df.groupby(subscriber_bins, observed=True)['ad_revenue_usd'].mean().to_frame('avg_revenue'))

## 7. Correlation and Outlier Detection

In [ ]:
corr = df[numeric_cols].corr()['ad_revenue_usd'].sort_values(ascending=False)
display(corr.to_frame('correlation_with_revenue'))

outlier_summary = []
for col in numeric_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((df[col] < low) | (df[col] > high)).sum()
    outlier_summary.append([col, low, high, count])
display(pd.DataFrame(outlier_summary, columns=['column', 'lower_bound', 'upper_bound', 'outlier_count']))

## 8. Data Cleaning and Feature Engineering

In [ ]:
import sys
sys.path.append('../src')
from preprocessing import clean_dataset
from feature_engineering import add_engineered_features

cleaned = clean_dataset(df)
prepared = add_engineered_features(cleaned)
prepared[['engagement_rate', 'likes_per_view', 'comments_per_view', 'watch_time_efficiency', 'interaction_score', 'subscriber_engagement_score']].head()

Created features:

- Engagement Rate = `(likes + comments) / views`
- Likes Per View = `likes / views`
- Comments Per View = `comments / views`
- Watch Time Efficiency = `watch_time_minutes / video_length_minutes`
- Interaction Score = `0.7 * likes + 1.3 * comments`
- Subscriber Engagement Score = `(likes + comments) / subscribers`

## 9. Model Comparison

In [ ]:
comparison = pd.read_csv('../reports/results/model_comparison.csv')
display(comparison)

The best model is **Lasso Regression** because it has the highest R2 score and the lowest error values. This is also easy to explain in a viva.

## 10. Model Interpretation and Business Insights

In [ ]:
importance = pd.read_csv('../reports/results/feature_importance.csv')
display(importance.head(10))
sample_predictions = pd.read_csv('../reports/results/sample_predictions.csv')
display(sample_predictions[['actual_revenue', 'predicted_revenue']].head())

Important business takeaways: watch time is the strongest driver, engagement features help explain viewer interest, and category/device/country analysis can guide content planning. The final Streamlit app converts these outputs into a prediction interface and revenue optimization advisor.